In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from simple_modflow.modflow.mf6.voronoiplus import TriangleGrid as Triangle
from shapely import Polygon
from simple_modflow.modflow.mf6.voronoiplus import VoronoiGridPlus as Vor
from pathlib import Path
from simple_modflow.modflow.mf6.simplemodel import SimpleModel
import simple_modflow.modflow.mf6.mfsimbase as simbase
from simple_modflow.modflow.mf6.boundaries import Boundaries
from simple_modflow.modflow.mf6.recharge import RechargeFromShp
from simple_modflow.modflow.mf6.drn import DRN
from simple_modflow.modflow.mf6.ghb import GHB
from simple_modflow.modflow.mf6.uzf import UZFPackageData
from simple_modflow.modflow.mf6.headsplus import HeadsPlus as hp
import simple_modflow.modflow.mf6.mfsimbase as mf
import shapely as shp
import bisect
import flopy
from simple_modflow.modflow.utils.datatypes.readers import read_shp_gpkg
import scipy
import pickle
import pandas as pd
import geopandas as gpd
import figs as f
from simple_modflow.modflow.mf6.mfsimbase import SimulationBase
from simple_modflow.modflow.utils.surfaces import InterpolatedSurface
from pandas import IndexSlice as idxx


C:\Users\lukem\AppData\Local\Programs\Python\Python312\Lib\site-packages\reportlab\lib\rl_safe_eval.py:12: DeprecationWarning: ast.NameConstant is deprecated and will be removed in Python 3.14; use ast.Constant instead
  haveNameConstant = hasattr(ast,'NameConstant')
C:\Users\lukem\AppData\Local\Programs\Python\Python312\Lib\site-packages\dash\_jupyter.py:30: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  _dash_comm = Comm(target_name="dash")


In [114]:
domain = Path(r"C:\Users\lukem\mf6\SSB data\ssb_domain.gpkg")
ssb_area = Path(r"C:\Users\lukem\mf6\SSB data\ssb_grid_refine.gpkg")
rch_polys = Path(r"C:\Users\lukem\mf6\SSB data\recharge_polys.gpkg")
lay1_botm = Path(r"C:\Users\lukem\mf6\SSB data\layer0.tif")
lay2_botm = Path(r"C:\Users\lukem\mf6\SSB data\layer1.tif")
lay3_botm = Path(r"C:\Users\lukem\mf6\SSB data\layer2.tif")
model_top = Path(r"C:\Users\lukem\QGIS\LIDAR\Tehaleh\Tehaleh_2020.tif")
wells = Path(r"C:\Users\lukem\mf6\SSB data\model_wells.gpkg")
k_layer1 = Path(r"C:\Users\lukem\mf6\SSB data\k_layer1.txt")
iheads_path = Path(r"C:\Users\lukem\mf6\SSB data\iheads.csv")
uics_path = Path(r"C:\Users\lukem\mf6\SSB data\ssb_UIC_gallery.gpkg")
low_k_extent_path = Path(r"C:\Users\lukem\mf6\SSB data\low_K_layer_extent.gpkg")
cfc_drn_path = Path(r"C:\Users\lukem\mf6\SSB data\boundaries\CFC_drain.gpkg")
uics_north_line_path = Path(r"C:\Users\lukem\mf6\SSB data\ssb_UIC_gallery_north_line.gpkg")

In [115]:
mjp_k_rasters = [Path(r"C:\Users\lukem\mf6\SSB data\Kx_layer1.tif"), Path(r"C:\Users\lukem\mf6\SSB data\Kz_layer1.tif")]
low_k_cells = model.vor.get_vor_cells_as_series(low_k_extent_path)[0]
mjp_ks = model.vor.get_raster_vals_at_centroids(mjp_k_rasters, labels=['kx', 'kz'])
mjp_ks.loc[mjp_ks.kx < 0, 'kx'] = 1
mjp_ks.loc[mjp_ks.kz < 0, 'kz'] = 1
kx = mjp_ks.kx
kz = mjp_ks.kz
# kxs = np.column_stack([kx for _ in range(nlay)])
# kzs = np.column_stack([kz for _ in range(nlay)])
kx1: pd.Series = kx.copy() * 5
kx1[kx1 > 200] = 1000
kz1 = kz.copy() * 5
kz1[kz1 > 200] = 1000

reading raster file C:\Users\lukem\mf6\SSB data\Kx_layer1.tif
reading raster file C:\Users\lukem\mf6\SSB data\Kz_layer1.tif


In [117]:
kx1.loc[(kx1.index.isin(low_k_cells)) & (kx1 > 60)] = 60
kx1

0        1000.000000
1        1000.000000
2        1000.000000
3         156.851366
4         105.745825
            ...     
22698    1000.000000
22699    1000.000000
22700      60.000000
22701    1000.000000
22702      60.000000
Name: kx, Length: 22703, dtype: float64

In [42]:
import pickle
with open(Path(r"C:\Users\lukem\mf6\ssb_no_inflow\ssb_no_inflow.model"), 'rb') as file:
    pre_model: SimulationBase = pickle.load(file)
with open(Path(r"C:\Users\lukem\mf6\ssb_N_linear\ssb_N_linear.model"), 'rb') as file:
    model: SimulationBase = pickle.load(file)

In [9]:
cfc_cells = pre_model.vor.get_vor_cells_as_series(read_shp_gpkg(cfc_drn_path).union_all())[0]
cfc_elevs = pre_model.vor.gdf_topbtm[0].loc[cfc_cells]
cfc_cells = cfc_elevs[cfc_elevs > 433].index.to_list()

Imported 1 features from C:\Users\lukem\mf6\SSB data\boundaries\CFC_drain.gpkg


In [44]:
kcells = model.vor.get_vor_cells_as_series(read_shp_gpkg(uics_north_line_path).union_all())[0]
pd.Series(model.gwf.npf.k[0].get_data()).loc[kcells]

Imported 23 features from C:\Users\lukem\mf6\SSB data\ssb_UIC_gallery_north_line.gpkg


132     200.0
133     200.0
134     200.0
135     200.0
136     200.0
        ...  
7963    200.0
7967    200.0
7969    200.0
7970    200.0
7971    200.0
Length: 5104, dtype: float64

In [45]:
import simple_modflow
ch: simple_modflow.modflow.utils.datatypes.choros.Choro = pre_model.cor(per=0, locs=wells, layer=0)
"""surf = model.surf.hds(per=14)
xs = surf.xy_meshgrid[0][0]
ys = surf.xy_meshgrid[1][:, 0]
zi_filled = np.where(np.isnan(surf.surface), 0, surf.surface)
ch.choropleth.add_contour(x=xs, y=ys,
                          z=zi_filled
)"""
ch.choropleth.show()

In [18]:
model.srf.hds(per=14, layer=4).save_raster('ssb_mounding_per14_lay5_asdesigned.tif')

In [62]:
model.vor.plot3d().show(renderer='browser')

In [ ]:
model.hds.plot_heads(3534, show_times=True)

In [10]:
df = pre_model.bud('drn').df
float(df.loc[idxx[cfc_cells, :], 'q'].sum()) / (24 * 60* 60)

-39.96939324259219

In [ ]:
below_top = (per1.loc[idxx[0, :], :].values - model.modelgrid.top.reshape((-1, 1))).flatten().tolist()

In [ ]:
hds: pd.DataFrame = model.hds.all_heads
pre_hds: pd.DataFrame = pre_model.hds.all_heads
per0 = pre_hds.loc[idxx[(9, 0), :, :], :].droplevel(0)
per1 = hds.loc[idxx[(19, 14), :, :], :].droplevel(0)
lyr0_diff = (per1-per0).loc[idxx[0, :], :].values.flatten().tolist()
lyr2_diff = (per1-per0).loc[idxx[2, :], :].values.flatten().tolist()
model.cor(per=14, locs=wells, layer=0, custom_zs=below_top).plot()

In [ ]:
model.srf.hds(per=29, layer=2).save_raster('ssb_mounding_test_30days_lay3.tif')

In [ ]:
model.hds.plot_heads(locs=test_wells+[174], layer=0)
model.hds.plot_heads(locs=test_wells, layer=1)
model.hds.plot_heads(locs=test_wells, layer=2)

In [ ]:
vor.choropleth(locs=wells).plot()

In [ ]:
heads: pd.DataFrame = model.hds.all_heads.loc[idxx[(9,0), :, :], :]
heads.droplevel(0).pivot_table(columns='layer', index='cell', values='elev').to_csv(iheads_path, index=False)

In [ ]:
ex = Path(r"C:\Users\lukem\mf6\Cumberland general\cumb_aq_test_logger_data.xlsx")
ex = pd.read_excel(ex, header=[0,1,2])
tw1: pd.DataFrame = ex.loc[:, idxx['TW-1 Constant Rate Test', :,['t\n(min)', 's\n(feet)']]]
col: pd.MultiIndex = tw1.columns
fig = AquiferTestFigure(plot_type='semilog')
for well in col.get_level_values(1).unique():
    xs = tw1.loc[:, idxx[:, well, 't\n(min)']].values.flatten().tolist()
    ys = tw1.loc[:, idxx[:, well, 's\n(feet)']].values.flatten().tolist()
    fig.add_scattergl(
        x=xs,
        y=ys,
        name=well
    )
fig.show()

In [ ]:
aq_tst = Path(r"C:\Users\lukem\Python\data\EB-118W test.xlsx")
pump_well = pd.read_excel(aq_tst).loc[:, ['min', 's']]
hand_data = pd.read_excel(aq_tst, sheet_name=1).loc[:, ['step', 'min', 'Q', 's feet']]
hand_data['step'] = hand_data['step'].ffill()
step_times = [hand_data[hand_data['step'] == stp].iloc[0].loc['min'] for stp in [1,2,3,4,5]]
step_times
def stp_times(x):
    step_idx = bisect.bisect(step_times, x) - 1
    return x - step_times[step_idx], step_idx
pump_well['step times'] = pump_well.loc[:, 'min'].apply(lambda x: stp_times(x)[0])
pump_well['step'] = pump_well.loc[:, 'min'].apply(lambda x: stp_times(x)[1]+1)

fig = f.Fig()
for stp in [1,2,3,4,5]:
    fig.add_scattergl(
        x=pump_well[pump_well['step'] == stp].loc[:, 'step times'],
        y=pump_well[pump_well['step'] == stp].loc[:, 's'],
        name=stp
    )
fig.update_layout(
    xaxis_type="log",
    #yaxis_type="log",
)
fig.show()
fig2 = f.Fig()
fig2.add_scattergl(
    y=hand_data['Q'].ffill().bfill(),
    x=hand_data['min'],
    name='Q'
)
fig2.add_scattergl(
    x=pump_well['min'],
    y=pump_well['s'],
)
fig2.show()

In [ ]:
from shiny.express import input, ui
from shinywidgets import render_plotly

ui.input_selectize(
    "var", "Select variable",
    choices=["bill_length_mm", "body_mass_g"]
)

@render_plotly
def hist():
    import plotly.express as px
    from palmerpenguins import load_penguins
    df = load_penguins()
    return px.histogram(df, x=input.var())

# Create the Shiny Express app
app = App(ui, None)


In [ ]:
from shiny.express import input, ui
from shinywidgets import render_plotly

ui.input_selectize(
    "var", "Select variable",
    choices=["bill_length_mm", "body_mass_g"]
)

@render_plotly
def hist():
    import plotly.express as px
    from palmerpenguins import load_penguins
    df = load_penguins()
    return px.histogram(df, x=input.var())

# Create the Shiny Express app
app = App(ui, None)
